# Fiddler Programmatic Dashboards Script (with Custom Chart Order)

Export segments, custom metrics, and charts from one Fiddler model and import them to another model, with support for:
* Cross-instance transfers (different Fiddler instances)
* Cross-model transfers (different models on same or different instances)
* Automatic schema validation with error handling for incompatible columns
* **Custom chart ordering on the new dashboard**

## How chart ordering works

After exporting charts from the source dashboard, a dedicated cell auto-populates `CHART_ORDER`
with the chart titles in their **current source layout order**. You can then rearrange that list
before running the import and dashboard-creation cells.

Charts are arranged in a **2-column grid** (left-to-right, top-to-bottom):
```
| position 1  | position 2  |
| position 3  | position 4  |
| position 5  | position 6  |
```

## Prerequisites

* Source Fiddler instance URL and API token
* Target Fiddler instance URL and API token (can be same as source)
* Source model must exist; **target model will be created automatically if it does not exist**
* `fiddler_utils` package installed (run cell below)

In [55]:
# Install dependencies
# %pip install -q fiddler-client

# Install fiddler_utils from parent directory
# If not already installed, run from repo root: pip install -e .
import sys

sys.path.insert(0, "..")

In [56]:
import fiddler as fdl
from fiddler_utils import (
    ConnectionManager,
    SegmentManager,
    CustomMetricManager,
    SchemaValidator,
)
from requests import HTTPError, Response
from fiddler.libs.http_client import RequestClient

print(f"Fiddler client version: {fdl.__version__}")
print("fiddler_utils: Successfully imported")

Fiddler client version: 3.10.0
fiddler_utils: Successfully imported


## Configuration

In [76]:
# Token
TOKEN = ""

In [ ]:
# Source Fiddler Instance
SOURCE_URL = ""  # e.g., 'https://source.fiddler.ai'
TARGET_TOKEN = TOKEN
SOURCE_PROJECT_NAME = "" # e.g. ABC
SOURCE_MODEL_NAME = "" # e.g. EmailClassifierML_100725_v2
SOURCE_MODEL_VERSION = ""  # Optional, leave empty for unversioned models

# Target Fiddler Instance (can be same as source - model must already exist)
TARGET_URL = ""  # e.g., 'https://target.fiddler.ai'
TARGET_TOKEN = TOKEN
TARGET_PROJECT_NAME = ""
TARGET_MODEL_NAME = "" # use same as source if you want to leverage the same data
TARGET_MODEL_VERSION = ""  # Optional, leave empty for unversioned models

# Asset Selection (empty lists = export all)
SEGMENTS_TO_EXPORT = []  # e.g., ['segment1', 'segment2'] or [] for all
CUSTOM_METRICS_TO_EXPORT = []  # e.g., ['metric1'] or [] for all


# Title for the new dashboard created on the target model
TARGET_DASHBOARD_TITLE = ""  # e.g., 'Email Classification Monitoring v2'

# Chart Export Options (for bonus section)
SOURCE_DASHBOARD_ID = ""  # Important - Dashboard UUID to export charts from 
CHART_IDS_TO_EXPORT = []  # Optional. e.g., ['uuid1', 'uuid2'] or [] for all

## Setup Connections and Initialize Managers

In [58]:
# Setup connection manager for handling multiple instances
conn_mgr = ConnectionManager(log_level="WARNING")
conn_mgr.add("source", url=SOURCE_URL, token=SOURCE_TOKEN)
conn_mgr.add("target", url=TARGET_URL, token=TARGET_TOKEN)

# Initialize asset managers
segment_mgr = SegmentManager()
metric_mgr = CustomMetricManager()

print("✓ Connection manager configured")
print("✓ Asset managers initialized")

✓ Connection manager configured
✓ Asset managers initialized


## Fetch Source and Target Models

Fetches the source model and attempts to fetch the target model.
If the target model does not exist yet, it is automatically created by duplicating
the source model's schema, spec, and task — so you don't need to set it up manually.

In [59]:
# Connect to source and get model
with conn_mgr.use("source"):
    source_project = fdl.Project.from_name(SOURCE_PROJECT_NAME)
    source_model_kwargs = {"project_id": source_project.id, "name": SOURCE_MODEL_NAME}
    if SOURCE_MODEL_VERSION:
        source_model_kwargs["version"] = SOURCE_MODEL_VERSION

    source_model = fdl.Model.from_name(**source_model_kwargs)
    print(f"Source model: {source_model.name} (ID: {source_model.id})")

# Connect to target and get (or create) model
with conn_mgr.use("target"):
    target_project = fdl.Project.from_name(TARGET_PROJECT_NAME)
    target_model_kwargs = {"project_id": target_project.id, "name": TARGET_MODEL_NAME}
    if TARGET_MODEL_VERSION:
        target_model_kwargs["version"] = TARGET_MODEL_VERSION

    try:
        # Attempt to fetch an existing target model
        target_model = fdl.Model.from_name(**target_model_kwargs)
        print(f"Target model: {target_model.name} (ID: {target_model.id})")

    except fdl.NotFound:
        # Target model does not exist — create it by duplicating the source model.
        #
        # duplicate() clones the source model's full schema, spec (inputs/outputs/targets),
        # task type, and task params. Only the name, project, and optional version are changed.
        # No data is copied — the new model starts empty and ready to receive events.
        print(f"Target model '{TARGET_MODEL_NAME}' not found — creating from source schema...")

        # Switch back to source connection to call duplicate() (it uses the active connection)
        with conn_mgr.use("source"):
            target_model = source_model.duplicate(
                version=TARGET_MODEL_VERSION if TARGET_MODEL_VERSION else None
            )

        # Re-point the duplicated model at the target project and name
        target_model.name = TARGET_MODEL_NAME
        target_model.project_id = target_project.id

        # Register the new model with the target Fiddler instance
        target_model.create()
        print(f"✓ Created target model: {target_model.name} (ID: {target_model.id})")

Source model: EmailClassifierML_100725_v2 (ID: e6d85806-4b04-43b1-b698-f86b0f570e26)
Target model: EmailClassifierML_100725_v2 (ID: e6d85806-4b04-43b1-b698-f86b0f570e26)


## Schema Comparison

Compare source and target model schemas to identify potential compatibility issues.

In [60]:
# Get schema information from both models
with conn_mgr.use("source"):
    source_columns = SchemaValidator.get_model_columns(source_model)
    print(f"Source model has {len(source_columns)} columns")

with conn_mgr.use("target"):
    target_columns = SchemaValidator.get_model_columns(target_model)
    print(f"Target model has {len(target_columns)} columns")

with conn_mgr.use("source"):
    comparison = SchemaValidator.compare_schemas(source_model, target_model)

print("\n" + "=" * 60)
print("SCHEMA COMPARISON")
print("=" * 60)
print(f"\nCommon columns: {len(comparison.in_both)}")

if comparison.only_in_source:
    print(
        f"\n⚠️  Columns in SOURCE but MISSING in TARGET ({len(comparison.only_in_source)}):"
    )
    for col in sorted(comparison.only_in_source):
        col_info = source_columns[col]
        print(f"  - {col} ({col_info.role}, {col_info.data_type})")
else:
    print("\n✅ No missing columns - all source columns exist in target")

if comparison.only_in_target:
    print(
        f"\n📋 Columns in TARGET but NOT in SOURCE ({len(comparison.only_in_target)}):"
    )
    for col in sorted(list(comparison.only_in_target)[:5]):
        col_info = target_columns[col]
        print(f"  - {col} ({col_info.role}, {col_info.data_type})")
    if len(comparison.only_in_target) > 5:
        print(f"  ... and {len(comparison.only_in_target) - 5} more")

if comparison.type_mismatches:
    print(f"\n⚠️  Data type DIFFERENCES ({len(comparison.type_mismatches)}):")
    for col, (source_type, target_type) in list(comparison.type_mismatches.items())[:5]:
        print(f"  - {col}: source={source_type} → target={target_type}")
else:
    print("\n✅ All common columns have matching data types")

print(
    f"\n{'✅' if comparison.is_compatible else '⚠️ '} Schema compatibility: {comparison.is_compatible}"
)
print("=" * 60)

Source model has 30 columns
Target model has 30 columns

SCHEMA COMPARISON

Common columns: 30

✅ No missing columns - all source columns exist in target

✅ All common columns have matching data types

✅ Schema compatibility: True


## Export Segments

Export segments from source model using `SegmentManager`.

In [61]:
print("\n" + "=" * 60)
print("EXPORTING SEGMENTS")
print("=" * 60)

with conn_mgr.use("source"):
    # Export segments (filtered by name if specified)
    exported_segments = segment_mgr.export_assets(
        model_id=source_model.id,
        names=SEGMENTS_TO_EXPORT if SEGMENTS_TO_EXPORT else None,
    )

    print(f"\n✓ Exported {len(exported_segments)} segment(s)")

    for seg_data in exported_segments:
        print(f"\n  Segment: {seg_data.name}")
        print(f"    Definition: {seg_data.data['definition']}")
        print(f"    Referenced columns: {seg_data.referenced_columns}")


EXPORTING SEGMENTS

✓ Exported 4 segment(s)

  Segment: Email Bounce Detection
    Definition: "FDL Email Bounce Detection (prompt)" == 'Match'
    Referenced columns: {'FDL Email Bounce Detection (prompt)'}

  Segment: Feedback_Complaint Segment
    Definition: "predicted" == 'feedback_complaint'
    Referenced columns: {'predicted'}

  Segment: Irrelevant LLM Response
    Definition: not "FDL Answer Relevance result"
    Referenced columns: {'FDL Answer Relevance result'}

  Segment: Return segment
    Definition: "predicted" == 'returns'
    Referenced columns: {'predicted'}


## Import Segments

Import segments to target model with automatic validation.

In [62]:
print("\n" + "=" * 60)
print("IMPORTING SEGMENTS")
print("=" * 60)

with conn_mgr.use("target"):
    # Import with validation and error handling
    segment_result = segment_mgr.import_assets(
        target_model_id=target_model.id,
        assets=exported_segments,
        validate=True,
        dry_run=False,
        skip_invalid=True,
        overwrite=False,
    )

    print("\nResults:")
    print(f"  ✅ Successfully imported: {segment_result.successful}")
    print(f"  🔄 Skipped (existing): {segment_result.skipped_existing}")
    print(f"  ⊘  Skipped (invalid): {segment_result.skipped_invalid}")
    print(f"  ❌ Failed: {segment_result.failed}")

    if segment_result.errors:
        print("\n  Errors:")
        for name, error in segment_result.errors:
            print(f"    - {name}: {error}")


IMPORTING SEGMENTS

Results:
  ✅ Successfully imported: 0
  🔄 Skipped (existing): 4
  ⊘  Skipped (invalid): 0
  ❌ Failed: 0


## Export Custom Metrics

Export custom metrics from source model using `CustomMetricManager`.

In [63]:
print("\n" + "=" * 60)
print("EXPORTING CUSTOM METRICS")
print("=" * 60)

with conn_mgr.use("source"):
    # Export custom metrics (filtered by name if specified)
    exported_metrics = metric_mgr.export_assets(
        model_id=source_model.id,
        names=CUSTOM_METRICS_TO_EXPORT if CUSTOM_METRICS_TO_EXPORT else None,
    )

    print(f"\n✓ Exported {len(exported_metrics)} custom metric(s)")

    for metric_data in exported_metrics:
        print(f"\n  Metric: {metric_data.name}")
        print(f"    Definition: {metric_data.data['definition']}")
        print(f"    Referenced columns: {metric_data.referenced_columns}")

        # Show complexity info
        metadata = metric_data.data["metadata"]
        metric_type = "Aggregation" if metadata["is_aggregation"] else "Simple"
        print(f"    Type: {metric_type}")
        if metadata["functions_used"]:
            print(f"    Functions: {', '.join(metadata['functions_used'])}")


EXPORTING CUSTOM METRICS

✓ Exported 0 custom metric(s)


## Import Custom Metrics

Import custom metrics to target model with automatic validation.

In [64]:
print("\n" + "=" * 60)
print("IMPORTING CUSTOM METRICS")
print("=" * 60)

with conn_mgr.use("target"):
    # Import with validation and error handling
    metric_result = metric_mgr.import_assets(
        target_model_id=target_model.id,
        assets=exported_metrics,
        validate=True,
        dry_run=False,
        skip_invalid=True,
        overwrite=False,
    )

    print("\nResults:")
    print(f"  ✅ Successfully imported: {metric_result.successful}")
    print(f"  🔄 Skipped (existing): {metric_result.skipped_existing}")
    print(f"  ⊘  Skipped (invalid): {metric_result.skipped_invalid}")
    print(f"  ❌ Failed: {metric_result.failed}")

    if metric_result.errors:
        print("\n  Errors:")
        for name, error in metric_result.errors:
            print(f"    - {name}: {error}")


IMPORTING CUSTOM METRICS

Results:
  ✅ Successfully imported: 0
  🔄 Skipped (existing): 0
  ⊘  Skipped (invalid): 0
  ❌ Failed: 0


## Summary Report

In [65]:
print("\n" + "=" * 60)
print("EXPORT/IMPORT SUMMARY")
print("=" * 60)

print("\nSegments:")
print(f"  Total exported: {len(exported_segments)}")
print(f"  Successfully imported: {segment_result.successful}")
print(f"  Skipped (existing): {segment_result.skipped_existing}")
print(f"  Skipped (invalid): {segment_result.skipped_invalid}")
print(f"  Failed: {segment_result.failed}")

print("\nCustom Metrics:")
print(f"  Total exported: {len(exported_metrics)}")
print(f"  Successfully imported: {metric_result.successful}")
print(f"  Skipped (existing): {metric_result.skipped_existing}")
print(f"  Skipped (invalid): {metric_result.skipped_invalid}")
print(f"  Failed: {metric_result.failed}")

total_success = segment_result.successful + metric_result.successful
total_skipped_existing = (
    segment_result.skipped_existing + metric_result.skipped_existing
)
total_skipped_invalid = segment_result.skipped_invalid + metric_result.skipped_invalid
total_failed = segment_result.failed + metric_result.failed
total_exported = len(exported_segments) + len(exported_metrics)

print("\n" + "=" * 60)
print(f"OVERALL: {total_success}/{total_exported} assets successfully imported")
if total_skipped_existing > 0:
    print(f"  {total_skipped_existing} skipped (already exist on target)")
if total_skipped_invalid > 0:
    print(f"  {total_skipped_invalid} skipped (validation errors)")
if total_failed > 0:
    print(f"  {total_failed} failed during import")
print("=" * 60)


EXPORT/IMPORT SUMMARY

Segments:
  Total exported: 4
  Successfully imported: 0
  Skipped (existing): 4
  Skipped (invalid): 0
  Failed: 0

Custom Metrics:
  Total exported: 0
  Successfully imported: 0
  Skipped (existing): 0
  Skipped (invalid): 0
  Failed: 0

OVERALL: 0/4 assets successfully imported
  4 skipped (already exist on target)


## Export/Import Charts

Demonstrate cross-instance chart transfer using `ChartManager`.

This section shows how fiddler_utils simplifies chart transfers.

**To use this section:**
1. Set `SOURCE_DASHBOARD_ID` in the configuration cell above (find dashboard ID in Fiddler UI URL)
2. OR set `CHART_IDS_TO_EXPORT` to manually specify chart UUIDs
3. Run the cells below to export and import charts

**Note:** Chart API uses unofficial Fiddler endpoints and may change without notice.

In [66]:
from fiddler_utils import ChartManager

print("\n" + "=" * 60)
print("INITIALIZING CHART MANAGERS")
print("=" * 60)

# Create separate managers for source and target
# Each needs its own URL/token for RequestClient
source_chart_mgr = ChartManager(url=SOURCE_URL, token=SOURCE_TOKEN)
target_chart_mgr = ChartManager(url=TARGET_URL, token=TARGET_TOKEN)

print("\n✓ Source ChartManager initialized")
print("✓ Target ChartManager initialized")


INITIALIZING CHART MANAGERS

✓ Source ChartManager initialized
✓ Target ChartManager initialized


In [67]:
print("\n" + "=" * 60)
print("EXPORTING CHARTS")
print("=" * 60)

# Determine export method
if SOURCE_DASHBOARD_ID:
    print(f"\nExporting charts from dashboard: {SOURCE_DASHBOARD_ID}")
elif CHART_IDS_TO_EXPORT:
    print(f"\nExporting {len(CHART_IDS_TO_EXPORT)} charts by ID")
else:
    print("\n⚠️  No SOURCE_DASHBOARD_ID or CHART_IDS_TO_EXPORT specified.")
    print("   Set one of these in the configuration cell to export charts.")
    exported_charts = []

if SOURCE_DASHBOARD_ID or CHART_IDS_TO_EXPORT:
    with conn_mgr.use("source"):
        try:
            # Export charts using dashboard_id or chart_ids.
            # When using SOURCE_DASHBOARD_ID, charts are returned in the order
            # they appear in the source dashboard layout.
            exported_charts = source_chart_mgr.export_charts(
                dashboard_id=SOURCE_DASHBOARD_ID if SOURCE_DASHBOARD_ID else None,
                chart_ids=CHART_IDS_TO_EXPORT if CHART_IDS_TO_EXPORT else None,
            )

            print(f"\n✓ Exported {len(exported_charts)} chart(s)\n")

            # Display exported chart details
            for i, chart in enumerate(exported_charts, 1):
                print(f"{i}. {chart.get('title', 'Untitled')}")
                print(f"   Type: {chart.get('query_type', 'unknown')}")

                # Show data source info
                data_source = chart.get("data_source", {})
                queries = data_source.get("queries", [])

                if queries:
                    # Show first query details for monitoring charts
                    query = queries[0]
                    metric_info = []

                    if "baseline_name" in query:
                        metric_info.append(f"baseline={query['baseline_name']}")
                    if query.get("metric_type") == "custom":
                        metric_info.append(
                            f"custom_metric={query.get('metric', 'N/A')}"
                        )
                    if "segment" in query and query["segment"]:
                        metric_info.append(f"segment={query['segment']}")

                    if metric_info:
                        print(f"   Dependencies: {', '.join(metric_info)}")
                print()
        except Exception as e:
            print(f"\n❌ Failed to export charts: {e}")
            exported_charts = []


EXPORTING CHARTS

Exporting charts from dashboard: 5d92c435-92d4-45ed-84f9-949d4b799d9c

✓ Exported 13 chart(s)

1. Summary View
   Type: ANALYTICS

2. Email Classification Accuracy
   Type: MONITORING
   Dependencies: segment={'id': '12d33096-c231-42b9-938e-8c1bb41024ff'}

3. Confusion Matrix 
   Type: ANALYTICS

4. Email Category by BU
   Type: ANALYTICS

5. Data Integrity Violations 
   Type: MONITORING

6. Traffic
   Type: MONITORING

7. Email Drift based on Centroid Distance
   Type: MONITORING

8. Email Drift Visualized
   Type: EMBEDDING

9. Answer Relevance - Fiddler Trust Model
   Type: MONITORING
   Dependencies: segment={'id': '46d6cb52-90a6-4b6b-8420-cdb81a622b42'}

10. DNS Error Detection
   Type: MONITORING
   Dependencies: segment={'id': 'cccce02e-fdb2-4c40-b49c-2fa10f8a1dfb'}

11. Email Sentiment Classification
   Type: MONITORING

12. LLM Observability
   Type: ANALYTICS

13. Prompt Safety Metrics
   Type: ANALYTICS



## Set Dashboard Chart Order

Charts from the source dashboard are printed with a number when exported above.
Use those numbers in `CHART_ORDER` to control which charts appear on the new dashboard and in what order.

**To set the order:**
1. Run the first cell below — it prints all available charts with their position numbers.
2. Edit `CHART_ORDER` in the second cell, using those numbers in your desired sequence.
3. Run the second cell to see a grid preview of the resulting layout — the numbered reference stays visible above it.
4. Repeat steps 2–3 until satisfied, then continue to the import cells.

**Rules:**
- Only charts whose number appears in `CHART_ORDER` will be included — others are excluded.
- Leave `CHART_ORDER = []` to include all charts in their original source order.
- Duplicate numbers and out-of-range numbers will be flagged.

In [68]:
# Prints the source dashboard charts with their position numbers and a grid preview.
# Use the numbers shown here when setting CHART_ORDER in the cell below.

exported_titles = [chart.get("title", "Untitled") for chart in exported_charts]
n = len(exported_titles)

if not exported_titles:
    print("⚠️  No charts found. Check that SOURCE_DASHBOARD_ID or CHART_IDS_TO_EXPORT is set.")
else:
    print("=" * 60)
    print("AVAILABLE CHARTS")
    print("=" * 60)
    for i, title in enumerate(exported_titles, 1):
        print(f"  {i:>2}. {title}")
    print()
    print("Set CHART_ORDER in the cell below using these numbers, then re-run that cell")
    print("to see a preview of the resulting dashboard layout.")


AVAILABLE CHARTS
   1. Summary View
   2. Email Classification Accuracy
   3. Confusion Matrix 
   4. Email Category by BU
   5. Data Integrity Violations 
   6. Traffic
   7. Email Drift based on Centroid Distance
   8. Email Drift Visualized
   9. Answer Relevance - Fiddler Trust Model
  10. DNS Error Detection
  11. Email Sentiment Classification
  12. LLM Observability
  13. Prompt Safety Metrics

Set CHART_ORDER in the cell below using these numbers, then re-run that cell
to see a preview of the resulting dashboard layout.


In [75]:
# Step 1: Enter your preferred chart order using the numbers from the cell above.
#         Any charts you don't include here will be appended at the end (see Step 2).
#         To exclude charts entirely, remove them from both lines.
#         Leave empty [] to keep all charts in their original source order.
#
MY_CHART_ORDER = [7,1,4,5]

# Step 2: Any charts not listed in MY_CHART_ORDER are automatically appended at the end.
#         Leave this line as-is — it fills in the rest for you.
#
CHART_ORDER = MY_CHART_ORDER + [i for i in range(1, n + 1) if i not in MY_CHART_ORDER]

# ── Resolve numbers to titles and validate ────────────────────────────────────

if not exported_titles:
    print("⚠️  No charts found. Run the export cell first.")
else:
    if not CHART_ORDER:
        # Empty list → include all charts in original source order
        final_chart_titles = exported_titles
    else:
        # Validate: flag out-of-range numbers
        out_of_range = [x for x in CHART_ORDER if not (1 <= x <= n)]
        if out_of_range:
            print(f"⚠️  Out-of-range chart numbers (valid range is 1–{n}): {out_of_range}")

        # Validate: flag duplicates
        seen, duplicates = set(), []
        for x in CHART_ORDER:
            if x in seen:
                duplicates.append(x)
            seen.add(x)
        if duplicates:
            print(f"⚠️  Duplicate chart numbers (each number should appear once): {duplicates}")

        # Resolve valid numbers to titles (preserve the order specified)
        final_chart_titles = [
            exported_titles[x - 1] for x in CHART_ORDER
            if 1 <= x <= n
        ]

    # ── Grid preview ─────────────────────────────────────────────────────────
    # Column width: wide enough for the longest title + number prefix + padding
    num_prefix_len = 5  # e.g. " 1.  "
    col_width = max((len(t) for t in final_chart_titles), default=0) + num_prefix_len + 1
    col_width = max(col_width, 24)  # minimum readable width

    h_div  = "─" * col_width
    top    = f"┌{h_div}┬{h_div}┐"
    mid    = f"├{h_div}┼{h_div}┤"
    bottom = f"└{h_div}┴{h_div}┘"

    def padded(pos, text):
        """Format a cell as ' {pos}. {title}' left-aligned to col_width."""
        if text:
            content = f"{pos}. {text}"
        else:
            content = ""
        return f" {content:<{col_width - 1}}"

    print("\nDashboard layout preview:\n")
    rows = [final_chart_titles[i:i+2] for i in range(0, len(final_chart_titles), 2)]
    pos = 1
    for r, row in enumerate(rows):
        print(top if r == 0 else mid)
        left  = padded(pos,     row[0]) if len(row) > 0 else padded("", "")
        right = padded(pos + 1, row[1]) if len(row) > 1 else padded("", "")
        print(f"│{left}│{right}│")
        pos += 2
    print(bottom)
    print(f"\n{len(final_chart_titles)} chart(s) selected.")
    print("\n" + "─" * 60)
    print("If this looks correct, continue to the next cell to import charts.")
    print("To adjust: edit MY_CHART_ORDER above and re-run this cell.")
    print("─" * 60)



Dashboard layout preview:

┌────────────────────────────────────────────┬────────────────────────────────────────────┐
│ 1. Email Drift based on Centroid Distance  │ 2. Summary View                            │
├────────────────────────────────────────────┼────────────────────────────────────────────┤
│ 3. Email Category by BU                    │ 4. Data Integrity Violations               │
├────────────────────────────────────────────┼────────────────────────────────────────────┤
│ 5. Email Classification Accuracy           │ 6. Confusion Matrix                        │
├────────────────────────────────────────────┼────────────────────────────────────────────┤
│ 7. Traffic                                 │ 8. Email Drift Visualized                  │
├────────────────────────────────────────────┼────────────────────────────────────────────┤
│ 9. Answer Relevance - Fiddler Trust Model  │ 10. DNS Error Detection                    │
├────────────────────────────────────────────┼──────

In [70]:
if not exported_charts:
    print("\n⊘ No charts to import. Skipping import.")
    chart_result = {"successful": 0, "failed": 0, "errors": []}
else:
    print("\n" + "=" * 60)
    print("IMPORTING CHARTS")
    print("=" * 60)

    with conn_mgr.use("target"):
        # Perform actual import
        chart_result = target_chart_mgr.import_charts(
            target_project_id=target_project.id,
            target_model_id=target_model.id,
            charts=exported_charts,
            validate=True,
            dry_run=False,  # Actually create charts
        )

        print("\nResults:")
        print(f"  ✅ Successfully imported: {chart_result['successful']}")
        print(f"  ❌ Failed: {chart_result['failed']}")

        if chart_result.get("errors"):
            print(f"\n  Errors encountered:")
            for title, error in chart_result["errors"]:
                print(f"    • {title}")
                print(f"      {error}")

        # Update overall summary
        if chart_result["successful"] > 0:
            print(
                f"\n✓ Successfully imported {chart_result['successful']} chart(s) to target model"
            )

        imported_charts = chart_result['imported_charts']


IMPORTING CHARTS

Results:
  ✅ Successfully imported: 13
  ❌ Failed: 0

✓ Successfully imported 13 chart(s) to target model


## Package Charts Into a New Default Dashboard

Creates a new dashboard on the target model using the chart order defined in `CHART_ORDER` above.
The dashboard is automatically set as the default dashboard for the target model.

In [71]:
client = RequestClient(
    TARGET_URL,
    headers={
        "Content-Type": "application/json",
        "Authorization": f"Bearer {TARGET_TOKEN}",
    },
)

def create_dashboard(
    project_name: str,
    model_name: str,
    charts: list,
    dashboard: dict,
) -> dict:
    dashboards_url = "v2/dashboards"

    dashboard["organization_name"] = fdl.conn.organization_name
    dashboard["project_name"] = project_name

    try:
        # Map chart titles to their UUIDs from the imported charts
        chart_title_to_uuid = {chart["title"]: chart["id"] for chart in charts}
        model_name = dashboard.get("model_name", None)
        dashboard.pop("model_name", None)

        # Resolve chart_title placeholders in the layout to real UUIDs
        for saved_chart in dashboard.get("layouts"):
            chart_title = saved_chart.get("chart_title")
            saved_chart["chart_uuid"] = chart_title_to_uuid.get(chart_title)
            saved_chart.pop("chart_title", None)

        dashboard_resp: Response = client.post(url=dashboards_url, data=dashboard)

        project = fdl.Project.get_or_create(name=project_name)
        model = fdl.Model.from_name(name=model_name, project_id=project.id)

        # Set the created dashboard as default for this model
        default_dashboard_url = f"v3/models/{model.id}/default-dashboard"
        payload = {"dashboard_uuid": dashboard_resp.json()["data"].get("uuid")}
        client.put(url=default_dashboard_url, data=payload)

        return dashboard_resp

    except HTTPError as hex:
        print(
            f"HTTPError occured: {hex.response.text} with error code {hex.response.status_code}"
        )
        raise hex


def generate_chart_layout(chart_title, x, y):
    """Build a single layout entry for one chart at grid position (x, y)."""
    return {
        "chart_title": chart_title,
        "grid_props": {
            "height": 1,
            "position_x": x,
            "position_y": y,
            "width": 1,
        },
    }


def generate_chart_layouts(chart_titles):
    """Arrange chart titles into a 2-column grid layout (left-to-right, top-to-bottom)."""
    chart_layouts = []
    for i, chart_title in enumerate(chart_titles):
        x = i % 2   # column: 0 = left, 1 = right
        y = i // 2  # row: increments every 2 charts
        chart_layouts.append(generate_chart_layout(chart_title, x, y))
    return chart_layouts


def update_default_dashboard(chart_titles):
    """Create a new dashboard from the given chart title order and set it as default."""
    dashboard = {
        "layouts": generate_chart_layouts(chart_titles),
        "model_name": TARGET_MODEL_NAME,
        "options": {
            "filters": {"time_label": "6m", "time_zone": "America/Los_Angeles"}
        },
        "organization_name": fdl.conn.organization_name,
        "project_name": TARGET_PROJECT_NAME,
        "title": TARGET_DASHBOARD_TITLE,
    }

    dashboard_response = create_dashboard(
        project_name=TARGET_PROJECT_NAME,
        model_name=TARGET_MODEL_NAME,
        charts=imported_charts,
        dashboard=dashboard,
    )

    project = fdl.Project.from_name(TARGET_PROJECT_NAME)
    model = fdl.Model.from_name(name=TARGET_MODEL_NAME, project_id=project.id)

    # Set the created dashboard as default
    default_dashboard_url = f"v3/models/{model.id}/default-dashboard"
    payload = {"dashboard_uuid": dashboard_response.json()['data']['uuid']}
    default_dashboard_response = client.put(url=default_dashboard_url, data=payload)

    if default_dashboard_response.status_code == 200:
        print(f"  ✅ Successfully created new default dashboard")


# final_chart_titles was resolved in the "Set Dashboard Chart Order" cell above.
# It contains exactly the charts to include, in the order specified by CHART_ORDER.
# Run that cell again if you want to adjust the selection before creating the dashboard.

update_default_dashboard(final_chart_titles)

  ✅ Successfully created new default dashboard
